In [ ]:
import pandas as pd
import numpy as np
fov3 = pd.read_parquet("../data/processed/fov3_strips.parquet")
# Get all unique gene targets in the panel
panel_genes = set(fov3['target'].unique())
print(f"Total genes in panel: {len(panel_genes)}")

In [ ]:
from liana.resource import select_resource # liana Python package (Dimitrov et al., 2022)

# CellChatDB: curated L-R database (Jin et al., Nat Comms 2021)
# mentioned in connectomeDB2025 - lists LIANA+'s consensus resource as a simple CSV on github (python native)
cellchat_lr = select_resource('CellChatDB')
print(f"CellChatDB: {len(cellchat_lr)} ligand-receptor pairs")
print(f"Unique ligands: {cellchat_lr['ligand'].nunique()}")
print(f"Unique receptors: {cellchat_lr['receptor'].nunique()}")

In [ ]:
# Get panel genes
panel_genes = set(fov3['target'].unique())
print(f"Panel genes: {len(panel_genes)}")

# Filter to pairs where BOTH ligand and receptor are in our panel
mask = (cellchat_lr['ligand'].isin(panel_genes) & 
        cellchat_lr['receptor'].isin(panel_genes))
lr_in_panel = cellchat_lr[mask].copy().reset_index(drop=True)

print(f"L-R pairs with both genes in panel: {len(lr_in_panel)}")
print()
print(lr_in_panel.to_string())

In [ ]:
# Check transcript counts per strip for each gene in our L-R pairs
lr_genes = set(lr_in_panel['ligand']) | set(lr_in_panel['receptor'])

for strip_name in ['strip_2', 'strip_3']:
    strip_df = fov3[fov3['strip'] == strip_name]
    counts = strip_df['target'].value_counts()
    
    print(f"\n{strip_name}:")
    for gene in sorted(lr_genes):
        n = counts.get(gene, 0)
        flag = '  OK' if n >= 50 else '  LOW' if n > 0 else '  ABSENT'
        print(f"  {gene:>12}: {n:>4} transcripts {flag}")

**Results:**
- Every single L-R pair on CellCHatDB was below the 50 transcript threshold for meaningful K-results
- The counts were majoritively housekeeping genes so it makes sense
- We are only looking at a single strip of a single FOV atm
- 

In [ ]:
import spatialdata as sd
sdata_s1 = sd.read_zarr("../data/raw/updated_stitched_S1.zarr")
s1_pts = sdata_s1.points['points'].compute() # lazy Dask to eager Pandas DF

In [ ]:
print(s1_pts.columns.tolist())
print(s1_pts.head(2))

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# --- Configuration ---
fov_col = 'fov'
x_col = 'x_global_px'
y_col = 'y_global_px'
seed = 42
fov_ids = sorted(s1_pts[fov_col].unique())

print(f"Found {len(fov_ids)} FOVs in S1\n")

for fov_id in fov_ids:
    fov_df = s1_pts[s1_pts[fov_col] == fov_id]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].scatter(fov_df[x_col], fov_df[y_col],
                    s=0.5, alpha=0.3, color='steelblue')
    axes[0].set_aspect('equal')
    axes[0].set_title(f'Spatial layout')
    axes[0].set_xlabel(x_col)
    axes[0].set_ylabel(y_col)
    
    axes[1].hist(fov_df[x_col], bins=100, color='steelblue', alpha=0.7)
    axes[1].set_title(f'X-coordinate distribution')
    axes[1].set_xlabel(x_col)
    axes[1].set_ylabel('Count')
    
    plt.suptitle(f'FOV {fov_id}  (n={len(fov_df):,} transcripts)', fontsize=13)
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
fov_config = {
   # 1: {'n_strips': 3, 'keep': [1, 2, 3]},
   # 2: {'n_strips': 3, 'keep': [1, 2, 3]},
   # 3: {'n_strips': 3, 'keep': [1, 2, 3]},
    4: {'n_strips': 3, 'keep': [1, 2, 3]},
    5: {'n_strips': 3, 'keep': [1, 2, 3]},
   # 6: {'n_strips': 3, 'keep': [1, 2, 3]},
   # 7: {'n_strips': 3, 'keep': [1, 2, 3]},
    8: {'n_strips': 3, 'keep': [1, 2, 3]},
    9: {'n_strips': 3, 'keep': [1, 2, 3]},
    10: {'n_strips': 3, 'keep': [1, 2, 3]},
    11: {'n_strips': 3, 'keep': [1, 2, 3]},
   # 12: {'n_strips': 2, 'keep': [1, 3]},
   # 13: {'n_strips': 1, 'keep': [1]}
}

In [ ]:
from sklearn.mixture import GaussianMixture

labelled_fovs = []

for fov_id, config in fov_config.items():
    fov_df = s1_pts[s1_pts[fov_col] == fov_id].copy()
    n_strips = config['n_strips']
    
    # Fit GMM
    X = fov_df[[x_col]].values
    gmm = GaussianMixture(n_components=n_strips, random_state=42)
    gmm.fit(X)
    
    raw_labels = gmm.predict(X)
    
    # Reorder by ascending x: strip_1 = leftmost
    means = gmm.means_.flatten()
    order = np.argsort(means)
    label_map = {old: new for new, old in enumerate(order)}
    ordered_labels = np.array([label_map[l] for l in raw_labels])
    
    fov_df['strip'] = ['strip_' + str(l + 1) for l in ordered_labels]
    
    # Filter to kept strips
    strips_to_keep = [f'strip_{i}' for i in config['keep']]
    fov_df = fov_df[fov_df['strip'].isin(strips_to_keep)].copy()
    
    # Visualise
    colors = {'strip_1': 'steelblue', 'strip_2': 'tomato', 'strip_3': 'seagreen'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for strip_name in sorted(fov_df['strip'].unique()):
        mask = fov_df['strip'] == strip_name
        axes[0].hist(fov_df.loc[mask, x_col], bins=80, alpha=0.6,
                     color=colors[strip_name], label=strip_name)
        axes[1].scatter(fov_df.loc[mask, x_col], fov_df.loc[mask, y_col],
                        s=0.5, alpha=0.3, color=colors[strip_name],
                        label=strip_name)
    
    axes[0].set_title('GMM assignment — histogram')
    axes[0].set_xlabel(x_col)
    axes[0].legend()
    axes[1].set_title('GMM assignment — spatial')
    axes[1].set_aspect('equal')
    axes[1].legend(markerscale=5)
    leg = axes[1].legend(markerscale=5)
    for lh in leg.legend_handles:
        lh.set_alpha(1.0)
    
    plt.suptitle(f'FOV {fov_id} — {n_strips} strips '
                 f'(keeping {strips_to_keep})', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    print(fov_df['strip'].value_counts().sort_index().to_string())
    print()
    
    labelled_fovs.append(fov_df)

print(f"\nProcessed {len(labelled_fovs)} FOVs, "
      f"{sum(len(df) for df in labelled_fovs):,} total transcripts")

In [ ]:
# Combine all labelled FOVs
all_labelled = pd.concat(labelled_fovs, ignore_index=True)

# Overview plot
fig, ax = plt.subplots(figsize=(16, 12))
strip_colors = {'strip_1': 'steelblue', 'strip_2': 'tomato', 'strip_3': 'seagreen'}

for strip_name, color in strip_colors.items():
    mask = all_labelled['strip'] == strip_name
    if mask.any():
        ax.scatter(all_labelled.loc[mask, x_col],
                   all_labelled.loc[mask, y_col],
                   s=0.5, alpha=0.2, color=color, label=strip_name)

ax.set_aspect('equal')
ax.legend(markerscale=10, fontsize=12)
leg = ax.legend(markerscale=5)
for lh in leg.legend_handles:
    lh.set_alpha(1.0)
ax.set_title(f'All S1 FOVs — strip assignment overview\n'
             f'{len(all_labelled):,} transcripts across '
             f'{all_labelled[fov_col].nunique()} FOVs', fontsize=14)
plt.tight_layout()
plt.show()

# Save
all_labelled.attrs = {}
all_labelled.to_parquet('../data/processed/s1_all_strips.parquet', index=False)
print(f"Saved {len(all_labelled):,} transcripts across "
      f"{all_labelled[fov_col].nunique()} FOVs")

In [ ]:
print(all_labelled.head())

### L-R feasibility check on expanded dataset (all S1 FOVs)

In [ ]:
# Check L-R pairs again with the full dataset (all_labelled) instead of just fov3
expanded_panel_genes = set(all_labelled['target'].unique())
mask = (cellchat_lr['ligand'].isin(panel_genes) & 
        cellchat_lr['receptor'].isin(panel_genes))

lr_in_expanded_panel = cellchat_lr[mask].copy().reset_index(drop=True)

print(f"(Total genes in fov3 subset: {len(panel_genes)})")
print(f"L-R pairs with both genes in expanded panel: {len(expanded_panel_genes)}")

print(f"\nNumber passing min trancript count of {MIN_TRANSCRIPTS}: \n\n\tfov3: {len(lr_in_panel)}  vs. Expanded panel: {len(lr_in_expanded_panel)}")


In [ ]:

# Same threshold logic as single-FOV check, now applied to all_labelled.
# Pairs that pass here are candidates for K-function analysis in notebook 08.

MIN_TRANSCRIPTS = 50  # minimum per gene per strip for reliable K estimates
lr_genes = set(lr_in_expanded_panel['ligand']) | set(lr_in_expanded_panel['receptor'])

print("Feasibility check: L-R genes across all S1 FOVs")
print(f"Total transcripts: {len(all_labelled):,}")
print(f"Strips: {sorted(all_labelled['strip'].unique())}\n")

for strip_name in ['strip_1', 'strip_2', 'strip_3']:
    strip_df = all_labelled[all_labelled['strip'] == strip_name]
    counts = strip_df['target'].value_counts()
    
    print(f"{strip_name}  (n={len(strip_df):,} transcripts):")
    ok_genes = []
    for gene in sorted(lr_genes):
        n = counts.get(gene, 0)
        flag = 'OK' if n >= MIN_TRANSCRIPTS else 'LOW' if n > 0 else 'ABSENT'
        if flag == 'OK':
            ok_genes.append(gene)
        print(f"  {gene:>15}: {n:>5} transcripts  {flag}")
    print(f"\n  → {len(ok_genes)} genes pass threshold in {strip_name}: {ok_genes}\n")

### Compact summary: which L-R genes pass threshold in each strip ─────────

In [ ]:
# Identify viable L-R pairs for K-function analysis 
# A pair is viable only if BOTH ligand AND receptor meet the transcript threshold in a given strip

# Pre-compute per-strip gene counts once
strip_counts = {
    strip: all_labelled[all_labelled['strip'] == strip]['target'].value_counts()
    for strip in ['strip_1', 'strip_2', 'strip_3']
}

rows = []
for _, row in lr_in_expanded_panel.iterrows():
    lig, rec = row['ligand'], row['receptor']
    result = {'ligand': lig, 'receptor': rec}
    for strip, counts in strip_counts.items():
        n_lig = counts.get(lig, 0)
        n_rec = counts.get(rec, 0)
        result[f'{strip}_ligand_n'] = n_lig
        result[f'{strip}_receptor_n'] = n_rec
        result[f'{strip}_viable'] = (n_lig >= MIN_TRANSCRIPTS) and (n_rec >= MIN_TRANSCRIPTS)
    rows.append(result)

pair_df = pd.DataFrame(rows)

# Filter to pairs viable in at least one strip
any_viable = pair_df[['strip_1_viable', 'strip_2_viable', 'strip_3_viable']].any(axis=1)
viable_pairs = pair_df[any_viable].reset_index(drop=True)

print(f"L-R pairs viable (both partners n≥{MIN_TRANSCRIPTS}) in at least one strip:")
print(f"{len(viable_pairs)} of {len(lr_in_expanded_panel)} pairs pass\n")
print(viable_pairs[['ligand', 'receptor', 
                     'strip_1_viable', 'strip_2_viable', 
                     'strip_3_viable']].to_string())

In [ ]:
all_viable = viable_pairs[['strip_1_viable', 'strip_2_viable', 'strip_3_viable']].all(axis=1)
print(f"\n{all_viable.sum()} pairs are viable in ALL strips (ideal for cross-strip K analysis)")

viable_pairs[all_viable].to_parquet("../data/processed/viable_lr_pairs_all_strips.parquet")

#### We don't want to test all 146. 

Running K-functions on every viable pair is statistically indefensible - we'd need to correct for 146 comparisons and your results would be uninterpretable - need a principled selection strategy.

The sensible approach is to pick pairs based on biological hypothesis, not just data availability. 
We have three categories worth considering:

**Infection-relevant pairs** — chemokine signalling (CXCL9/CXCR3, CXCL10/CXCR3, CCL5/CCR5) are canonical immune recruitment axes that should be elevated in infected tissue. These are the primary candidates.

**Immune checkpoint pairs** — CD274/PDCD1 (PD-L1/PD-1) is directly relevant to infection response and immunosuppression.

**Vascular/growth factor pairs** — VEGFA/KDR, HGF/MET — less directly infection-relevant, useful as biological comparators.